<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1 | Representing Text as Tensors</h3>

We can use different approaches when representing text:

- **Character-level representation**<br/>
We represent text by treating each character as a number. If there are C unique characters in our text dataset, each character becomes a vector of lenght C consisting entirely of 0s but for a single 1 at the position of that character's index. For the word Hello, stacking the 5 individual one-hot encoded vectors, creates a tensor(matrix) with shape C x 5.

- **Word-level representation**<br/>
We create a vocabulary of all words in our text, and then represent words using one-hot encoding. This is better than character-level representation because each letter by itself does not have much meaning. By using higher level semantic concepts (words) we simplify the task for the neural network. But for a large dictionary size, we need to deal with high-dimensional sparse tensors.

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">1 | Text Classification Task</h3>

The AG_NEWS dataset we'll use, contains news headlines. We'll classify these headlines into one of four categories:
- World
- Sports
- Business
- Sci/Tech

In [ ]:
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
tf.get_logger().setLevel("ERROR")
keras = tf.keras

dataset = tfds.load("ag_news_subset")

In [2]:
ds_train = dataset["train"]
ds_test = dataset["test"]

print(f"Length of train dataset = {len(ds_train):,}")
print(f"Length of test dataset = {len(ds_test):,}")

Length of train dataset = 120,000
Length of test dataset = 7,600


Print the first five news headlines from our dataset:<br/>
Each headline contains the fields:
- label
- title
- description

In [3]:
classes = ["World", "Sports", "Business", "Sci/Tech"]

for i, x in zip(range(5), ds_train):
    print(f"{x["label"]} ({classes[x["label"]]}) -> {x["title"]} {x["description"]}")

3 (Sci/Tech) -> b'AMD Debuts Dual-Core Opteron Processor' b'AMD #39;s new dual-core Opteron chip is designed mainly for corporate computing applications, including databases, Web services, and financial transactions.'
1 (Sports) -> b"Wood's Suspension Upheld (Reuters)" b'Reuters - Major League Baseball\\Monday announced a decision on the appeal filed by Chicago Cubs\\pitcher Kerry Wood regarding a suspension stemming from an\\incident earlier this season.'
2 (Business) -> b'Bush reform may have blue states seeing red' b'President Bush #39;s  quot;revenue-neutral quot; tax reform needs losers to balance its winners, and people claiming the federal deduction for state and local taxes may be in administration planners #39; sights, news reports say.'
3 (Sci/Tech) -> b"'Halt science decline in schools'" b'Britain will run out of leading scientists unless science education is improved, says Professor Colin Pillinger.'
1 (Sports) -> b'Gerrard leaves practice' b'London, England (Sports Network

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">2 | Text Vectorization</h3>

To achieve word-level representation, we must:
- Use a **tokenizer** to split text into **tokens**.
- Build a **vocabulary** of these tokens.

So, we use the **TextVecorization** layer and the `adapt` method that goes through all text, and builds the vocabulary.

<h4 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">2.1 | Limiting Vocabulary Size</h4>

Since the vocabulary size is very large (> 100k words), it makes sense to remove worlds that are rarely present in the text. To limit the vocabulary size, use the `max_tokens` argument:

In [4]:
# Limit vocabulary size: Maximum number of unique words
vocab_size = 50_000
# Create the text vectorization layer. By default this
# 1. converts text to lower case,
# 2. removes punctuation and
# 3. splits sentences based on white spaces.
vectorizer = keras.layers.TextVectorization(max_tokens=vocab_size)
# Only use a subset of the whole dataset: Combine titles and descriptions
# from the first 500 samples of the dataset and adapt(train) the layer to these:
vectorizer.adapt(ds_train.take(500).map(lambda x: x["title"] + " " + x["description"]))

Access the actual vocabulary:

In [5]:
vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)

print(f"Length of vocabulary : {vocab_size}")
print(vocab[:10])

Length of vocabulary : 5335
['', '[UNK]', np.str_('the'), np.str_('to'), np.str_('a'), np.str_('in'), np.str_('of'), np.str_('and'), np.str_('on'), np.str_('for')]


Use the tokenizer to encode any text, into a set of numbers:

In [6]:
vectorizer("I love to play with my words")

<tf.Tensor: shape=(7,), dtype=int64, numpy=array([ 112, 3695,    3,  304,   11, 1041,    1])>

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">3 | Bag-of-Words Text Representation</h3>

**Bag-of-Words (BoW)** is the most simple to understand traditional vector representation. It's based on the fact that we can usually figure out the meaning of text, by just looking at the individual words - regardless of their order.

It treats documents as a simple collection—or "bag"—of individual words, completely disregarding 
- grammatical structures,
- sentence context, and 
- the original chronological order of the words.

Using **BoW**, each word is linked to a vector index, and a vector element contains the count of occurences of each word in a document.

In [7]:
# Generate a BoW representation using the Scikit-Learn library:
sc_vectorizer = CountVectorizer()
corpus = [
    "I like hot dogs.",
    "The dog ran fast.",
    "Its hot outside.",
]
# Learn the vocabulary dictionary (fit) and return document-term matrix (transform).
sc_vectorizer.fit_transform(corpus)
sc_vectorizer.transform(["My dog likes hot dogs on a hot day."]).toarray()

array([[1, 1, 0, 2, 0, 0, 0, 0, 0]])

In [8]:
# Same using the keras vectorizer
def to_bow(text):
    return tf.reduce_sum(tf.one_hot(vectorizer(text), vocab_size), axis=0)


to_bow("My dog likes hot dogs on a hot day.").numpy()

array([0., 5., 0., ..., 0., 0., 0.], shape=(5335,), dtype=float32)

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">4 | Training the BoW Classifier</h3>

**1. Convert our dataset to a BoW representation:**

In [9]:
batch_size = 128

ds_train_bow = ds_train.map(lambda x: (to_bow(x["title"] + x["description"]), x["label"])).batch(batch_size)

ds_test_bow = ds_test.map(lambda x: (to_bow(x["title"] + x["description"]), x["label"])).batch(batch_size)

**2. Define a simple classifier neural network that contains one linear layer.**<br/>

The input size is`vocab_size`.<br/>
The output size is 4 (the number of classes)<br/>
Final activation function is `softmax`, since this is a classification problem.

In [10]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(vocab_size,)),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(ds_train_bow, validation_data=ds_test_bow, shuffle=False)

938/938 ━━━━━━━━━━━━━━━━━━━━ 27s 28ms/step - accuracy: 0.8426 - loss: 0.6141 - val_accuracy: 0.8708 - val_loss: 0.4411


<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">5 | Training a Classifier as one Network</h3>

Define a network that includes our vectorizer (a Keras layer) and train it end-to-end.

In [11]:
def extract_text(x):
    return tf.strings.join([x["title"], x["description"]], separator=" ")


def tupelize(x):
    return (extract_text(x), x["label"])


class BowLayer(tf.keras.layers.Layer):
    def __init__(self, vectorizer, vocab_size, **kwargs):
        super().__init__(**kwargs)
        self.vectorizer = vectorizer
        self.vocab_size = vocab_size

    def call(self, x):
        tokens = self.vectorizer(x)
        return tf.reduce_sum(tf.one_hot(tokens, self.vocab_size), axis=1)


inp = tf.keras.Input(shape=(1,), dtype=tf.string)
x = BowLayer(vectorizer, vocab_size)(inp)
out = tf.keras.layers.Dense(4, activation="softmax")(x)
model = tf.keras.Model(inp, out)
model.summary()

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
    shuffle=False,
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bow_layer (BowLayer)            │ (None, 5335)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │        21,344 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,344 (83.38 KB)

 Trainable params: 21,344 (83.38 KB)

 Non-trainable params: 0 (0.00 B)

938/938 ━━━━━━━━━━━━━━━━━━━━ 59s 62ms/step - accuracy: 0.8383 - loss: 0.6088 - val_accuracy: 0.8749 - val_loss: 0.4201


So the model takes one string per example (InputLayer - (None, 1)), converts it to a 5,335-dimensional BoW vector, then predicts 4 class probabilities (Dense - (None, 4)).

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">6 | Bigrams, Trigrams and n-Grams</h3>

Generate a bigram BoW representation using scikit-learn:

In [12]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r"\b\w+\b", min_df=1)

corpus = [
    "I like hot dogs.",
    "The dog ran fast.",
    "Its hot outside.",
]

bigram_vectorizer.fit_transform(corpus)

print("Vocabulary:\n", bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(["My dog likes hot dogs on a hot day."]).toarray()

Vocabulary:
 {'i': 7, 'like': 11, 'hot': 4, 'dogs': 2, 'i like': 8, 'like hot': 12, 'hot dogs': 5, 'the': 16, 'dog': 0, 'ran': 14, 'fast': 3, 'the dog': 17, 'dog ran': 1, 'ran fast': 15, 'its': 9, 'outside': 13, 'its hot': 10, 'hot outside': 6}


array([[1, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">7 | Automatically Calculating BoW Vectors</h3>

Passing `output_mode="count"`, allows us to calculate the BoW vectors automatically!

In [13]:
model = keras.Sequential(
    [
        keras.layers.TextVectorization(max_tokens=vocab_size, output_mode="count"),
        keras.layers.Dense(4, activation="softmax"),
    ]
)
print("Training vectorizer")

model.layers[0].adapt(ds_train.take(500).map(extract_text))
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["acc"])
model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
    shuffle=False,
)

Training vectorizer
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - acc: 0.8501 - loss: 0.5915 - val_acc: 0.8783 - val_loss: 0.4164


<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">8 | Term Frequency - Inverse Document Frequency (TF-IDF)</h3>

TD-IDF is a variation of BoW, where instead of a binary 0/1 value indicating the appearance of a word in a document, a floating-point value is used, which is related to the frequency of the word occurence in the corpus.

$\Large w_{ij} = tf_{ij} \times log(\frac{N}{df_i})$

- $w_{ij}$  : the weight of a word i in the document j
- $tf_{ij}$ : number of occurences of i in j
- $N$       : number of documents in the collection
- $df_i$    : number of documents containing the word i in the whole collection.

Create TF-IDF vectorization of text with scikit-learn:

In [14]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
vectorizer.fit_transform(corpus)
vectorizer.transform(["My dog likes hot dogs on a hot day."]).toarray()

array([[0.43381609, 0.        , 0.43381609, 0.        , 0.65985664,
        0.43381609, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ]])

Same with `keras`. Remember to pass `output_mode="tf-idf"`

In [15]:
model = keras.Sequential(
    [
        keras.layers.TextVectorization(max_tokens=vocab_size, output_mode="tf_idf"),
        keras.layers.Dense(4, activation="softmax"),
    ]
)
print("Training vectorizer")
model.layers[0].adapt(ds_train.take(500).map(extract_text))
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["acc"])
model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
    shuffle=False,
)

Training vectorizer
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - acc: 0.8689 - loss: 0.4136 - val_acc: 0.8880 - val_loss: 0.3426
